Midterm 3- Assigment 2

Author: Filippo Alessandro Sandoval Villarreal matricola: 596689

in order to resolve this kind of task i will use a U-net a specific type of convolutional neural network that can helps for this type of problems.

In [33]:
# import that are useful for this task
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2
# fare import di pythorc o keras
import tensorflow as tf
from tensorflow import keras
from tensorflow import keras
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Activation, ReLU
from tensorflow.keras.layers import BatchNormalization, Conv2DTranspose, Concatenate
from tensorflow.keras.models import Model, Sequential

Andiamo ad usare keras per lo sviluppo della rete neurale ma prima andiamo a caricare il dataset

In [44]:

def load_image_mask_paths_cv2(image_dir, mask_dir):
    
    image_paths = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(".png")])
    mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir) if f.endswith(".png")])

    if len(image_paths) != len(mask_paths):
        raise ValueError("Il numero di immagini e maschere non corrisponde.")

    image_filenames = [os.path.splitext(os.path.basename(p))[0] for p in image_paths]
    mask_filenames = [os.path.splitext(os.path.basename(p))[0] for p in mask_paths]

    if image_filenames != mask_filenames:
        print("Attenzione: I nomi dei file immagine e maschera non corrispondono nello stesso ordine.")

    return image_paths, mask_paths

def load_and_preprocess_pair_cv2(image_path, mask_path, target_size=(256, 256)):
    """
    Carica una coppia immagine-maschera, le ridimensiona e le converte in array NumPy (con OpenCV).
    """
    image = cv2.imread(image_path, cv2.IMREAD_COLOR)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    resized_image = cv2.resize(image, target_size, interpolation=cv2.INTER_LINEAR)
    resized_mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)

    image_array = resized_image / 255.0  # Normalizza tra 0 e 1
    mask_array = resized_mask

    # Converti la maschera in un formato adatto alle etichette (es. 0 e 1)
    mask_array = (mask_array > 128).astype(np.uint8) # Esempio di sogliatura

    image = tf.convert_to_tensor(image_array)
    mask = tf.convert_to_tensor(mask_array)

    return image, mask

# Definisci i percorsi delle tue cartelle
image_directory = "weizmann_horse_db/horse/"  # Sostituisci con il percorso reale
mask_directory = "weizmann_horse_db/mask/"    # Sostituisci con il percorso reale

# Ottieni le liste dei percorsi dei file
image_paths, mask_paths = load_image_mask_paths_cv2(image_directory, mask_directory)

# Esempio di caricamento e preprocessing di una singola coppia
if image_paths and mask_paths:
    sample_image, sample_mask = load_and_preprocess_pair_cv2(image_paths[0], mask_paths[0])
    print("Forma dell'immagine:", sample_image.shape)
    print("Forma della maschera:", sample_mask.shape)

   

Forma dell'immagine: (256, 256, 3)
Forma della maschera: (256, 256)


In [45]:

images = []
masks = []
for i in range(0,len(image_paths)):
    image, mask = load_and_preprocess_pair_cv2(image_paths[i],mask_paths[i])
    images.append(image)
    masks.append(mask)


In [46]:
images[0]

<tf.Tensor: shape=(256, 256, 3), dtype=float64, numpy=
array([[[0.22745098, 0.29803922, 0.43137255],
        [0.21960784, 0.30588235, 0.41568627],
        [0.20784314, 0.30588235, 0.40392157],
        ...,
        [0.27843137, 0.47843137, 0.60784314],
        [0.25882353, 0.45882353, 0.58823529],
        [0.2627451 , 0.4627451 , 0.59215686]],

       [[0.23529412, 0.31372549, 0.43529412],
        [0.21176471, 0.30196078, 0.4       ],
        [0.19215686, 0.29803922, 0.38039216],
        ...,
        [0.28627451, 0.48627451, 0.61568627],
        [0.2627451 , 0.4627451 , 0.59215686],
        [0.25882353, 0.45882353, 0.58823529]],

       [[0.23137255, 0.31372549, 0.41568627],
        [0.20784314, 0.30196078, 0.38431373],
        [0.18431373, 0.29803922, 0.36470588],
        ...,
        [0.31372549, 0.51372549, 0.64313725],
        [0.27058824, 0.47058824, 0.6       ],
        [0.24313725, 0.44313725, 0.57254902]],

       ...,

       [[0.25882353, 0.33333333, 0.34509804],
        [0.25

Adesso provvediamo allo sviluppo della U network. First of all i am going to define th convolution fase the decoder and encode fase

In [56]:
def convolution_operation(entered_input, filters=64):
    # Taking first input and implementing the first conv block
    conv1 = Conv2D(filters, kernel_size = (3,3), padding = "same")(entered_input)
    batch_norm1 = BatchNormalization()(conv1)
    act1 = ReLU()(batch_norm1)
    
    # Taking first input and implementing the second conv block
    conv2 = Conv2D(filters, kernel_size = (3,3), padding = "same")(act1)
    batch_norm2 = BatchNormalization()(conv2)
    act2 = ReLU()(batch_norm2)
    
    return act2

def encoder(entered_input, filters=64):
    # Collect the start and end of each sub-block for normal pass and skip connections
    enc1 = convolution_operation(entered_input, filters)
    MaxPool1 = MaxPooling2D(strides = (2,2))(enc1)
    return enc1, MaxPool1

def decoder(entered_input, skip, filters=64):
    # Upsampling and concatenating the essential features
    Upsample = Conv2DTranspose(filters, (2, 2), strides=2, padding="same")(entered_input)
    Connect_Skip = Concatenate()([Upsample, skip])
    out = convolution_operation(Connect_Skip, filters)
    return out

so i define my u net

In [62]:
def U_Net(Image_Size):
    # Take the image size and shape
    input1 = Input(Image_Size)
    
    # Construct the encoder blocks
    skip1, encoder_1 = encoder(input1, 64)
    skip2, encoder_2 = encoder(encoder_1, 64*2)
    skip3, encoder_3 = encoder(encoder_2, 64*4)
    skip4, encoder_4 = encoder(encoder_3, 64*8)
    
    # Preparing the next block
    conv_block = convolution_operation(encoder_4, 64*16)
    
    # Construct the decoder blocks
    decoder_1 = decoder(conv_block, skip4, 64*8)
    decoder_2 = decoder(decoder_1, skip3, 64*4)
    decoder_3 = decoder(decoder_2, skip2, 64*2)
    decoder_4 = decoder(decoder_3, skip1, 64)
    
    out = Conv2D(1, 1, padding="same", activation="sigmoid")(decoder_4)

    model = Model(input1, out)
    return model

input_shape = (256, 256, 3)
model = U_Net(input_shape)
model.summary()
tf.keras.utils.plot_model(model, "model.png", show_shapes=False, show_dtype=False, show_layer_names=True, rankdir='TB', expand_nested=False, dpi=96)

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 conv2d_38 (Conv2D)          (None, 256, 256, 64)         1792      ['input_3[0][0]']             
                                                                                                  
 batch_normalization_36 (Ba  (None, 256, 256, 64)         256       ['conv2d_38[0][0]']           
 tchNormalization)                                                                                
                                                                                                  
 re_lu_36 (ReLU)             (None, 256, 256, 64)         0         ['batch_normalization_36

Adesso andiamo a definire un costum data generator

Una volta che abbiamo definito il modello ottengo